# Week 17, Day 1: Estimators: bias, variance and the sampling distribution

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w017/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: See a sampling distribution

A neuron fires as a Poisson process at 8 Hz. One experiment records 10 trials of 0.5 s and estimates the rate as total spikes / total time. Simulate 5,000 such experiments and store the estimates in `rate_hats`. Compare their spread with the formula sqrt(rate / (n T)).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1922)
rate, n, T = 8.0, 10, 0.5
counts = rng.poisson(rate * T, size=(5000, n))
rate_hats = ...
print(f"mean of estimates {rate_hats.mean():.3f} Hz, standard error {rate_hats.std():.3f} (formula {np.sqrt(rate / (n * T)):.3f})")
plt.hist(rate_hats, 40); plt.axvline(rate, color="k"); plt.xlabel("estimated rate (Hz)"); plt.show()

In [ ]:
import numpy as np
def test_shape():
    "rate_hats holds 5000 estimates"
    assert rate_hats.shape == (5000,)

def test_se():
    "the spread of the estimates matches sqrt(rate / (n T)) to within 5%"
    assert abs(rate_hats.std() - np.sqrt(8 / (10 * 0.5))) / np.sqrt(8 / (10 * 0.5)) < 0.05

def test_unbiased():
    "the estimates average to the true rate to within 0.1 Hz"
    assert abs(rate_hats.mean() - 8) < 0.1

test_shape()
test_se()
test_unbiased()
print("All checks passed")

### Exercise: Unbiased is not best

Draw 200,000 samples of size 5 from a Gaussian with variance 4. For the estimators that divide by n and by n − 1, compute the bias, variance and MSE, and store them as `results[name] = {'bias': ..., 'var': ..., 'mse': ...}`.

In [ ]:
x = rng.normal(0, 2, size=(200000, 5))
ss = ((x - x.mean(axis=1, keepdims=True)) ** 2).sum(axis=1)
results = {}
for name, d in [("n", 5), ("n-1", 4)]:
    est = ss / d
    results[name] = {"bias": ..., "var": ..., "mse": ...}
for name, r in results.items():
    print(f"divide by {name:3s}: bias {r['bias']:+.3f}, variance {r['var']:.3f}, MSE {r['mse']:.3f}")

In [ ]:
import numpy as np
def test_bias():
    "dividing by n is biased by about -sigma^2 / n; dividing by n - 1 is unbiased (within 0.05)"
    assert abs(results["n"]["bias"] + 4 / 5) < 0.05 and abs(results["n-1"]["bias"]) < 0.05

def test_decomposition():
    "for each estimator, MSE equals bias^2 + variance to within 1e-9"
    assert all(abs(r["mse"] - (r["bias"] ** 2 + r["var"])) < 1e-9 for r in results.values())

def test_mse():
    "the biased estimator has the smaller mean squared error"
    assert results["n"]["mse"] < results["n-1"]["mse"]

test_bias()
test_decomposition()
test_mse()
print("All checks passed")

### Exercise: The inflated peak

A neuron's mean counts at 8 orientations are given by `f`, peaking at 10 spikes. For 5,000 simulated experiments of 10 trials per orientation, estimate the peak two ways: `peak_naive` (the largest of the 8 means) and `peak_split` (choose the best orientation using trials 1 to 5, then report its mean over trials 6 to 10).

In [ ]:
f = np.array([2.0, 3.5, 6.0, 9.0, 10.0, 9.0, 6.0, 3.5])
true_peak = f.max()
counts = rng.poisson(f, size=(5000, 10, 8))         # experiments x trials x orientations
means = counts.mean(axis=1)
peak_naive = ...
best = counts[:, :5, :].mean(axis=1).argmax(axis=1)
peak_split = ...
print(f"true peak {true_peak};  naive estimate {peak_naive.mean():.2f};  split estimate {peak_split.mean():.2f}")

In [ ]:
import numpy as np
def test_inflated():
    "the max-of-means estimate of the peak is too high on average by more than 0.2 spikes"
    assert peak_naive.mean() - true_peak > 0.2

def test_split():
    "selecting on half the trials and measuring on the other half removes most of the bias (within 0.2)"
    assert abs(peak_split.mean() - true_peak) < 0.2 + 0.05 * true_peak

test_inflated()
test_split()
print("All checks passed")